# Day 178 — Prompt Engineering
**Month 10 | LangChain + MLflow + Evidently**

| Field | Value |
|-------|-------|
| Day | 178 |
| Topic | Prompt Engineering: Zero-shot, Few-shot, CoT, Role Prompting, PromptTemplate |
| Dataset | ReviewPulse India — 600 rows, seed=155 |
| LLM | Groq Free API — llama-3.1-8b-instant |
| Max Score | 90 points + 10★ Bonus |
| Stack | LangChain 0.2.16 + langchain-groq 0.1.9 |

---
**Learning Objectives**
1. Understand how prompt structure (zero-shot vs few-shot) affects LLM output quality
2. Apply Chain-of-Thought (CoT) prompting for structured reasoning
3. Use role/system prompts to control LLM persona and tone
4. Build reusable LangChain PromptTemplates with dynamic variables
5. Extract structured JSON output from LLMs reliably

---
## SECTION 1 — Environment Setup
**⚠️ Run this cell first. After install completes → Runtime → Restart Runtime → then run all remaining cells.**

In [1]:
# Pinned install — mandatory for Month 10
!pip install -q \
    langchain==0.2.16 \
    langchain-community==0.2.16 \
    langchain-groq==0.1.9 \
    groq==0.9.0 \
    httpx==0.27.0

print("Done — Restart Runtime now")

Done — Restart Runtime now


---
## SECTION 2 — Concept Notes
*(Read before starting tasks)*

### 2.1 — What is Prompt Engineering?

Prompt engineering is the practice of **structuring input text to an LLM** to reliably obtain a desired output. It is not magic — it is a systematic design discipline with measurable impact on output quality, consistency, and cost.

**Why it matters in production:**
- The same model gives dramatically different outputs depending on how the prompt is phrased
- Poor prompts = hallucinations, wrong formats, inconsistent results → client failures
- Good prompts = predictable, structured, auditable outputs → production-grade pipelines

---

### 2.2 — The Prompt Engineering Toolkit

| Technique | What It Does | When To Use |
|-----------|-------------|-------------|
| **Zero-shot** | No examples — just an instruction | Simple tasks, fast prototyping |
| **Few-shot** | 2–5 examples before the actual question | When zero-shot fails or quality matters |
| **Chain-of-Thought (CoT)** | Ask the model to reason step-by-step before answering | Complex classification, reasoning tasks |
| **Role/System Prompt** | Assign a persona: "You are a senior analyst..." | Controlling tone, domain focus, output style |
| **Output Format Control** | Specify JSON, bullet list, table structure | Downstream parsing and automation |
| **PromptTemplate** | Parameterised reusable prompts with `{variable}` slots | Production pipelines, batch processing |

---

### 2.3 — Temperature and Sampling

| Parameter | Effect | Recommended Value |
|-----------|--------|-------------------|
| `temperature=0.0` | Deterministic, always picks highest-prob token | Classification, JSON extraction |
| `temperature=0.3–0.7` | Moderate creativity | Summarisation, explanation |
| `temperature=1.0+` | High randomness, more creative | Brainstorming, creative writing |

**Rule of thumb:** Structured outputs → low temperature. Creative tasks → higher temperature.

---

### 2.4 — Prompt Anatomy

A well-structured prompt has up to 4 components:
```
[ROLE]       → Who the model is
[CONTEXT]    → Background information
[TASK]       → What to do
[FORMAT]     → How to respond
```
Not all prompts need all four — but production prompts usually need at least TASK + FORMAT.

---

### 2.5 — CoT Prompting (Chain-of-Thought)

Standard prompt: *"Classify this review as positive/negative/neutral."*

CoT prompt:
```
Analyse this review step by step:
Step 1: Identify key sentiment words
Step 2: Note any positive/negative indicators
Step 3: Consider the overall tone
Step 4: State your final classification
```

CoT forces the model to expose its reasoning, making errors visible and correctable. It also consistently improves accuracy on complex inputs.

---

### 2.6 — LangChain PromptTemplate

```python
from langchain.prompts import PromptTemplate

template = PromptTemplate(
    input_variables=["review_text", "category"],
    template="""
    You are a sentiment analyst. Classify the following {category} review.
    Review: {review_text}
    Respond in JSON: {{"sentiment": "positive/neutral/negative", "confidence": 0-1}}
    """
)

# Format with actual values
prompt_str = template.format(review_text="Great work!", category="Data Analysis")
```

**Note:** Double curly braces `{{}}` escape literal braces inside f-string style templates.

---
## SECTION 3 — Raw Data
**⚠️ DO NOT MODIFY THIS CELL — Source data must remain unchanged**

In [2]:
# ============================================================
# RAW DATA — DO NOT MODIFY
# ReviewPulse India | seed=155 | 600 rows
# ============================================================
import numpy as np
import pandas as pd

np.random.seed(155)
n = 600

categories = ['Web Development', 'Data Analysis', 'Graphic Design', 'Content Writing', 'SEO']
platforms  = ['Upwork', 'Fiverr', 'Freelancer', 'Toptal', 'PeoplePerHour']

positive_reviews = [
    "Excellent work, delivered on time and exceeded expectations completely.",
    "Outstanding freelancer, very professional and highly skilled in their domain.",
    "Perfect delivery, great communication throughout the entire project.",
    "Highly recommend this freelancer, quality work and fast turnaround.",
    "Amazing experience, will definitely hire again for future projects.",
    "Top-notch quality work, responsive and very easy to work with.",
    "Brilliant results, went above and beyond what was asked of them.",
]
neutral_reviews = [
    "Work was completed as requested, nothing exceptional but satisfactory.",
    "Decent quality, met the basic requirements but could improve communication.",
    "Average experience overall, delivered on time but nothing outstanding.",
    "Job was done adequately, some revisions were needed before final delivery.",
    "Acceptable work quality, freelancer was professional but lacked creativity.",
]
negative_reviews = [
    "Disappointed with the quality, did not meet the agreed specifications at all.",
    "Poor communication and missed multiple deadlines during the project.",
    "Work quality was below expectations, required significant rework afterward.",
    "Not satisfied with the results, freelancer was unresponsive to feedback.",
    "Would not recommend, multiple errors in the final deliverable.",
]

def assign_sentiment(rating):
    if rating >= 4.0:
        return 'positive'
    elif rating >= 2.5:
        return 'neutral'
    else:
        return 'negative'

def assign_review(rating, idx):
    s = assign_sentiment(rating)
    if s == 'positive':
        return positive_reviews[idx % len(positive_reviews)]
    elif s == 'neutral':
        return neutral_reviews[idx % len(neutral_reviews)]
    else:
        return negative_reviews[idx % len(negative_reviews)]

ratings = np.clip(np.random.normal(3.5, 1.2, n), 1, 5).round(1)
sentiments = [assign_sentiment(r) for r in ratings]
reviews    = [assign_review(ratings[i], i) for i in range(n)]

df = pd.DataFrame({
    'review_id':       range(1, n+1),
    'rating':          ratings,
    'category':        np.random.choice(categories, n),
    'platform':        np.random.choice(platforms, n),
    'review_text':     reviews,
    'word_count':      np.random.randint(10, 150, n),
    'sentiment_label': sentiments
})

print(f"Dataset shape: {df.shape}")
print(f"Sentiment distribution:\n{df['sentiment_label'].value_counts()}")
df.head()

Dataset shape: (600, 7)
Sentiment distribution:
sentiment_label
neutral     256
positive    221
negative    123
Name: count, dtype: int64


,review_id,rating,category,platform,review_text,word_count,sentiment_label
0,1,4.2,Data Analysis,Freelancer,"Excellent work, delivered on time and exceeded...",70,positive
1,2,3.8,Graphic Design,Upwork,"Decent quality, met the basic requirements but...",67,neutral
2,3,4.0,SEO,Upwork,"Perfect delivery, great communication througho...",91,positive
3,4,2.1,SEO,PeoplePerHour,"Not satisfied with the results, freelancer was...",59,negative
4,5,3.4,Web Development,Freelancer,"Acceptable work quality, freelancer was profes...",91,neutral


---
## SECTION 4 — LLM Setup

In [3]:
# ============================================================
# LLM SETUP — Groq Free API (using Colab Secrets)
# ============================================================
import os
from google.colab import userdata
from langchain_groq import ChatGroq

os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY")

llm = ChatGroq(
    model_name="llama-3.1-8b-instant",
    temperature=0,
    groq_api_key=os.environ["GROQ_API_KEY"]
)

test_response = llm.invoke("Reply with exactly: GROQ_OK")
print(f"LLM connectivity: {test_response.content}")

LLM connectivity: GROQ_OK


---
## SECTION 5 — Graded Tasks

| Task | Topic | Points |
|------|-------|--------|
| T1 | Zero-shot vs Few-shot Classification | 15 |
| T2 | Chain-of-Thought (CoT) Prompting | 20 |
| T3 | Role + System Prompt Engineering | 20 |
| T4 | LangChain PromptTemplate + Batch JSON | 20 |
| T5 | NRA Business Insight | 15 |
| **Total** | | **90** |
| ★ Bonus | Temperature A/B Test | 10★ |

---
### TASK 1 — Zero-shot vs Few-shot Sentiment Classification (15 pts)

**Business context:** You are building a ReviewPulse batch classifier. Before deploying it, you need to compare two prompting strategies on the same 10 reviews to choose the better approach.

**Steps:**
1. Select rows 0–9 from `df` as your test set (10 reviews)
2. Write a **zero-shot prompt** that instructs the LLM to classify sentiment as `positive`, `neutral`, or `negative`. No examples provided.
3. Write a **few-shot prompt** that provides **3 labelled examples** (one per class) before the actual review
4. Run **both prompts** on all 10 reviews using a `for` loop; store predictions in two lists: `zero_shot_preds` and `few_shot_preds`
5. Compare against `df['sentiment_label'].iloc[0:10]` — print accuracy for both approaches

**Requirements:**
- Both prompts must end with: `"Respond with exactly one word: positive, neutral, or negative"`
- Parse LLM output with `.content.strip().lower()` before storing
- Print a comparison DataFrame: `review_id | review_text | true_label | zero_shot | few_shot`
- Print accuracy score for both (correct / 10)

In [10]:
# ============================================================
# CELL  — IMPORTS
# Goal: Import all necessary libraries for prompt engineering.
# Method: Standard imports + LangChain Groq integration.
# ============================================================

import numpy as np
import pandas as pd
import os
import json
import re
from collections import Counter
import warnings
warnings.filterwarnings('ignore')

from langchain_groq import ChatGroq
from langchain.prompts import PromptTemplate

print("✅ All imports successful")

✅ All imports successful


In [11]:
# ============================================================
# TASK 1 — ZERO-SHOT vs FEW-SHOT SENTIMENT CLASSIFICATION
# Goal: Compare accuracy of zero-shot vs few-shot prompts on 10 reviews.
# Method: Define two functions, loop over test set, print comparison table.
# ============================================================

test_df = df.iloc[0:10].copy()

def zero_shot_classify(review_text):
    """Zero-shot: instruction only, no examples."""
    prompt = f"""
    Classify the following review as positive, neutral, or negative.
    Review: {review_text}
    Respond with exactly one word: positive, neutral, or negative
    """
    response = llm.invoke(prompt)
    return response.content.strip().lower()

def few_shot_classify(review_text):
    """Few-shot: 3 labelled examples (one per class)."""
    prompt = f"""
    Here are examples of review sentiment classification:

    Review: "Excellent work, delivered on time and exceeded expectations."
    Sentiment: positive

    Review: "Average experience, nothing special but got the job done."
    Sentiment: neutral

    Review: "Disappointed with the quality, did not meet specifications."
    Sentiment: negative

    Now classify this review:
    Review: {review_text}
    Respond with exactly one word: positive, neutral, or negative
    """
    response = llm.invoke(prompt)
    return response.content.strip().lower()

zero_shot_preds = []
few_shot_preds  = []

for _, row in test_df.iterrows():
    zs = zero_shot_classify(row['review_text'])
    fs = few_shot_classify(row['review_text'])
    zero_shot_preds.append(zs)
    few_shot_preds.append(fs)

test_df = test_df.assign(zero_shot=zero_shot_preds, few_shot=few_shot_preds)

print("=== T1: Zero-shot vs Few-shot Comparison ===")
print(test_df[['review_id', 'review_text', 'sentiment_label', 'zero_shot', 'few_shot']].to_string())

zs_acc = (test_df['zero_shot'] == test_df['sentiment_label']).mean()
fs_acc = (test_df['few_shot']  == test_df['sentiment_label']).mean()
print(f"\nZero-shot accuracy: {zs_acc:.1%}")
print(f"Few-shot accuracy:  {fs_acc:.1%}")

=== T1: Zero-shot vs Few-shot Comparison ===
   review_id                                                                    review_text sentiment_label zero_shot  few_shot
0          1        Excellent work, delivered on time and exceeded expectations completely.        positive  positive  positive
1          2    Decent quality, met the basic requirements but could improve communication.         neutral   neutral   neutral
2          3           Perfect delivery, great communication throughout the entire project.        positive  positive  positive
3          4       Not satisfied with the results, freelancer was unresponsive to feedback.        negative  negative  negative
4          5    Acceptable work quality, freelancer was professional but lacked creativity.         neutral   neutral   neutral
5          6  Disappointed with the quality, did not meet the agreed specifications at all.        negative  negative  negative
6          7               Brilliant results, went above an

---
### TASK 2 — Chain-of-Thought (CoT) Prompting (20 pts)

**Business context:** A low-confidence review has come in with a borderline rating (2.4). Standard zero-shot classification is unreliable on edge cases. Use CoT to force the model to reason step-by-step.

**Target row:** `df.iloc[50]` → `review_id=51`, `rating=2.4`, `sentiment_label='negative'`

**Steps:**
1. Retrieve `df.iloc[50]`
2. Build a **CoT prompt** with these exact 4 steps in the prompt text:
   - `Step 1: List all sentiment-bearing words or phrases in the review`
   - `Step 2: Identify whether each is positive, negative, or neutral`
   - `Step 3: Weigh the overall balance — which sentiment dominates?`
   - `Step 4: State your final classification (positive / neutral / negative)`
3. Run the CoT prompt on the target review; print the **full LLM response** (do not truncate)
4. Extract the final classification from the response. The last line of the LLM response should contain the word `positive`, `neutral`, or `negative` — use `.split()` or string search to extract it
5. Compare extracted classification to `sentiment_label` — print `MATCH ✅` or `MISMATCH ❌`

**Requirement:** The CoT prompt must contain the phrase `"step by step"` and all 4 step labels listed above.

In [12]:
# ============================================================
# TASK 2 — CHAIN-OF-THOUGHT PROMPTING
# Goal: Force reasoning on a borderline review (rating=2.4, negative).
# Method: Use a 4‑step CoT prompt, extract final classification from output.
# ============================================================

cot_row = df.iloc[50]  # review_id=51, rating=2.4, negative
print(f"Target review: review_id={cot_row['review_id']}, rating={cot_row['rating']}")
print(f"Text: {cot_row['review_text']}")
print(f"True label: {cot_row['sentiment_label']}\n")

cot_prompt = """
Analyse the following review step by step.

Review: {review_text}

Step 1: List all sentiment-bearing words or phrases in the review
Step 2: Identify whether each is positive, negative, or neutral
Step 3: Weigh the overall balance — which sentiment dominates?
Step 4: State your final classification (positive / neutral / negative)

Final classification:
"""

cot_response = llm.invoke(cot_prompt.format(review_text=cot_row['review_text']))
full_response = cot_response.content
print("=== CoT Full Response ===")
print(full_response)

# Extract the last occurrence of positive/neutral/negative
matches = re.findall(r'\b(positive|neutral|negative)\b', full_response.lower())
extracted_label = matches[-1] if matches else "unknown"

print(f"\nExtracted: {extracted_label}")
print(f"True label: {cot_row['sentiment_label']}")
if extracted_label == cot_row['sentiment_label']:
    print("Result: MATCH ✅")
else:
    print("Result: MISMATCH ❌")

Target review: review_id=51, rating=2.4
Text: Disappointed with the quality, did not meet the agreed specifications at all.
True label: negative

=== CoT Full Response ===
Let's analyze the review step by step.

**Step 1: List all sentiment-bearing words or phrases in the review**

1. Disappointed
2. quality
3. did not meet
4. agreed specifications
5. at all

**Step 2: Identify whether each is positive, negative, or neutral**

1. Disappointed - Negative
2. quality - Neutral (can be positive or negative depending on context, but in this case, it's used in a negative context)
3. did not meet - Negative
4. agreed specifications - Neutral (can be positive or negative depending on context, but in this case, it's used in a negative context)
5. at all - Negative (implies a strong negative sentiment)

**Step 3: Weigh the overall balance — which sentiment dominates?**

The review contains several negative sentiment-bearing words or phrases, including "Disappointed", "did not meet", and "at all"

---
### TASK 3 — Role + System Prompt Engineering (20 pts)

**Business context:** ReviewPulse is sold to freelance platforms. You want the LLM to behave like a **senior ReviewPulse Analyst** — not a generic AI — so outputs are domain-appropriate and client-ready.

**Steps:**
1. Build a **system prompt** that assigns this exact persona:
   > *"You are a senior ReviewPulse Analyst at a freelance intelligence platform. You specialise in analysing client reviews on platforms like Upwork, Fiverr, and Freelancer.com. Your tone is professional, data-driven, and concise. You always support observations with specific evidence from the review text."*
2. Build a **user prompt** that asks the LLM to produce a structured analysis of a review with these exact keys:
   - `sentiment`: positive / neutral / negative
   - `confidence`: low / medium / high
   - `key_phrases`: list of 2–3 notable phrases from the review
   - `client_recommendation`: one actionable sentence for the platform
3. Use `ChatGroq` with `temperature=0` and pass messages as `[("system", system_prompt), ("human", user_prompt)]`
4. Run on **row 5** of `df` (review_id=6, rating=1.6, negative review) and **row 8** (review_id=9, rating=4.5, positive review)
5. Print both full responses and verify each contains all 4 required keys

**Requirement:** The system prompt must use `llm.invoke()` with a list of tuples format — not a plain string.

In [21]:
# ============================================================
# TASK 3 — ROLE + SYSTEM PROMPT ENGINEERING (corrected)
# Goal: Assign a domain‑expert persona and request structured analysis.
# Method: Use system message + user prompt with 4 required keys.
# ============================================================

import re  # ensure re is imported (already in Cell 2)

row_neg = df.iloc[5]   # review_id=6, rating=1.6, negative
row_pos = df.iloc[8]   # review_id=9, rating=4.5, positive

print(f"Row 5: review_id={row_neg['review_id']}, rating={row_neg['rating']}, label={row_neg['sentiment_label']}")
print(f"Row 8: review_id={row_pos['review_id']}, rating={row_pos['rating']}, label={row_pos['sentiment_label']}")

# System prompt – exact persona text from spec
system_prompt = """You are a senior ReviewPulse Analyst at a freelance intelligence platform. You specialise in analysing client reviews on platforms like Upwork, Fiverr, and Freelancer.com. Your tone is professional, data-driven, and concise. You always support observations with specific evidence from the review text."""

def build_user_prompt(review_text, category, platform):
    """User prompt requesting the 4 structured keys."""
    return f"""
    Analyse this review from a {category} freelancer on {platform}.

    Review: {review_text}

    Provide a structured analysis with the following exact keys:
    - sentiment: positive / neutral / negative
    - confidence: low / medium / high
    - key_phrases: list of 2–3 notable phrases
    - client_recommendation: one actionable sentence for the platform

    Use the keys as headings and provide your analysis under each.
    """

def run_with_system_prompt(row):
    """Invoke LLM with system+human messages (tuple format)."""
    user_prompt = build_user_prompt(row['review_text'], row['category'], row['platform'])
    messages = [("system", system_prompt), ("human", user_prompt)]
    response = llm.invoke(messages)
    return response.content

response_neg = run_with_system_prompt(row_neg)
response_pos = run_with_system_prompt(row_pos)

print("=== T3: Negative Review Analysis ===")
print(response_neg)
print("\n=== T3: Positive Review Analysis ===")
print(response_pos)

# --- Robust key verification (fixes underscore vs space mismatch) ---
def normalize_text(text):
    """Remove markdown formatting and extra whitespace."""
    text = re.sub(r'\*+', '', text)          # remove asterisks
    text = re.sub(r'[#_\-]', ' ', text)       # replace some punctuation with spaces
    text = re.sub(r'\s+', ' ', text).strip()  # collapse multiple spaces
    return text.lower()

required_keys = ['sentiment', 'confidence', 'key_phrases', 'client_recommendation']
print("\nKey verification:")
for key in required_keys:
    # Convert underscore to space to match LLM's output style (e.g., "key phrases")
    search_key = key.replace('_', ' ')
    neg_ok = search_key in normalize_text(response_neg)
    pos_ok = search_key in normalize_text(response_pos)
    print(f"  {key}: neg={'✅' if neg_ok else '❌'}  pos={'✅' if pos_ok else '❌'}")

Row 5: review_id=6, rating=1.6, label=negative
Row 8: review_id=9, rating=4.5, label=positive
=== T3: Negative Review Analysis ===
**Sentiment:** Negative
The client explicitly states their disappointment with the quality of the work, indicating a negative sentiment.

**Confidence:** High
The client's statement is direct and unambiguous, leaving little room for interpretation. The use of the phrase "did not meet the agreed specifications at all" further reinforces the client's confidence in their negative assessment.

**Key Phrases:**
- "Disappointed with the quality"
- "did not meet the agreed specifications"
- "at all" (emphasizes the extent of the failure to meet expectations)

**Client Recommendation:** 
The client should consider providing detailed specifications and clear expectations to freelancers to avoid similar disappointments in the future.

=== T3: Positive Review Analysis ===
**Sentiment:** Positive

The review expresses a strong positive sentiment towards the freelancer,

---
### TASK 4 — LangChain PromptTemplate + Batch JSON Output (20 pts)

**Business context:** ReviewPulse needs to process reviews in batch and return machine-readable JSON for downstream storage in a database. Build a reusable PromptTemplate and run it over 10 reviews.

**Target rows:** `df.iloc[100:110]` (10 reviews — rows 100–109)

**Steps:**
1. Import `PromptTemplate` from `langchain.prompts`
2. Build a `PromptTemplate` with `input_variables=["review_text", "category", "rating"]` that instructs the LLM to respond **only** in this JSON format (no other text):
   ```json
   {"sentiment": "positive/neutral/negative", "confidence": 0.0-1.0, "flag_for_review": true/false}
   ```
   - `flag_for_review` must be `true` if rating < 2.5 (high-risk review)
   - Include in the prompt: *"Respond ONLY with valid JSON. No explanation, no markdown, no extra text."*
3. Run the template on `df.iloc[100:110]` using a `for` loop
4. Parse each response with `json.loads()` inside a `try/except` — if parsing fails, store `{"error": "parse_failed"}`
5. Build a results DataFrame with columns: `review_id | rating | true_label | predicted_sentiment | confidence | flag_for_review | parse_status`
6. Print the results DataFrame and count how many reviews were flagged (`flag_for_review == True`)

**Requirements:**
- Use `PromptTemplate.format()` to build the final prompt string for each row
- Use `json.loads()` for parsing (not `eval()`)
- `parse_status` = `'ok'` if parsed successfully, `'error'` if exception raised

In [14]:
# ============================================================
# TASK 4 — PROMPTTEMPLATE + BATCH JSON OUTPUT
# Goal: Build a reusable prompt that returns valid JSON for 10 reviews.
# Method: Use PromptTemplate, json.loads() inside try/except, build results DataFrame.
# ============================================================

batch_df = df.iloc[100:110].copy()
print(f"Batch size: {len(batch_df)} reviews")
print(batch_df[['review_id', 'rating', 'sentiment_label', 'review_text']].to_string())

# PromptTemplate with 3 input variables
json_template = PromptTemplate(
    input_variables=["review_text", "category", "rating"],
    template="""
    You are a sentiment analysis bot. For the following review, respond ONLY with valid JSON.

    Review: {review_text}
    Category: {category}
    Rating: {rating}

    The JSON must have exactly these keys:
    - "sentiment": "positive", "neutral", or "negative"
    - "confidence": a number between 0.0 and 1.0
    - "flag_for_review": true if rating < 2.5, false otherwise

    Respond ONLY with valid JSON. No explanation, no markdown, no extra text.
    """
)

results = []

for _, row in batch_df.iterrows():
    prompt_str = json_template.format(
        review_text=row['review_text'],
        category=row['category'],
        rating=row['rating']
    )
    response = llm.invoke(prompt_str)
    raw = response.content.strip()

    try:
        parsed = json.loads(raw)
        parse_status = 'ok'
    except Exception:
        parsed = {'error': 'parse_failed'}
        parse_status = 'error'

    results.append({
        'review_id': row['review_id'],
        'rating': row['rating'],
        'true_label': row['sentiment_label'],
        'predicted_sentiment': parsed.get('sentiment', None) if parse_status == 'ok' else None,
        'confidence': parsed.get('confidence', None) if parse_status == 'ok' else None,
        'flag_for_review': parsed.get('flag_for_review', None) if parse_status == 'ok' else None,
        'parse_status': parse_status
    })

results_df = pd.DataFrame(results)
print("\n=== T4: Batch JSON Results ===")
print(results_df.to_string())
flagged_count = results_df['flag_for_review'].sum()
print(f"\nReviews flagged for review: {flagged_count}")

Batch size: 10 reviews
     review_id  rating sentiment_label                                                                    review_text
100        101     1.0        negative  Disappointed with the quality, did not meet the agreed specifications at all.
101        102     4.8        positive            Highly recommend this freelancer, quality work and fast turnaround.
102        103     2.6         neutral         Average experience overall, delivered on time but nothing outstanding.
103        104     4.5        positive                 Top-notch quality work, responsive and very easy to work with.
104        105     3.2         neutral    Acceptable work quality, freelancer was professional but lacked creativity.
105        106     3.1         neutral         Work was completed as requested, nothing exceptional but satisfactory.
106        107     4.7        positive  Outstanding freelancer, very professional and highly skilled in their domain.
107        108     3.0         ne

---
### TASK 5 — NRA Business Insight (15 pts)

**Business context:** Write a ReviewPulse Analytics brief for a freelance platform product manager.

**Data anchor (lock these numbers — read from your printed outputs):**
- Overall dataset: 600 reviews, avg rating = **3.42**, positive = **36.83%**, negative = **20.50%**
- T4 batch (rows 100–109): use your actual `flagged_count` from Task 4
- T1: use your actual accuracy numbers from Task 1

**Requirements — write 3 NRA bullets:**

**NRA Format (mandatory):**
- **N (Number):** Single anchor stat — one number, not a range. Read directly from cell output.
- **R (Reason):** Causal mechanism — *why* does this number exist? Not a description of the outcome.
- **A (Action):** Specific, committed action with concrete parameters. No hedging.

**Three required insights:**
1. Platform health insight (use overall sentiment %)
2. Prompt strategy insight (compare zero-shot vs few-shot accuracy from T1)
3. Risk flagging insight (use T4 flag_for_review count)

**Format in a markdown cell:**
```
### Insight 1 — [Title with number]
N: ...
R: ...
A: ...
```

**── TASK 5 — YOUR NRA INSIGHTS HERE ──**

### Insight 1 — Platform Health (Sentiment Distribution)
**N:** 20.50% of 600 reviews are negative (from Cell 3 output).
**R:** The negative share indicates that approximately 1 in 5 freelancer engagements result in dissatisfaction – likely due to mismatched expectations, poor communication, or quality issues. This is a leading indicator of client churn on freelance platforms.
**A:** Set up a weekly sentiment dashboard that triggers a platform‑wide review if the negative sentiment rate exceeds 25% for two consecutive weeks. Also implement a post‑project feedback loop that automatically flags any new negative review for immediate client follow‑up.

### Insight 2 — Prompt Strategy Impact (Zero‑shot vs Few‑shot)
**N:** Zero‑shot and few‑shot both achieve 100.0% accuracy on the 10‑review test set – a difference of 0% (from T1 output).
**R:** The simple, clearly polarised nature of the reviews means the instruction alone is sufficient for perfect classification. Few‑shot examples do not add value here because the decision boundary is unambiguous.
**A:** Deploy the zero‑shot prompt as the production classifier to minimise token cost and latency. Maintain a weekly validation pipeline that re‑runs this comparison on a held‑out set; if zero‑shot accuracy drops below 95%, switch to few‑shot or add a CoT layer.

### Insight 3 — Risk Flagging (Flag_for_review from T4)
**N:** In the batch of 10 reviews (rows 100–109), 2 reviews were flagged (`flag_for_review=True`) because rating < 2.5 (from T4 output).
**R:** Ratings below 2.5 correlate with unresolved delivery failures – clients explicitly state unmet specifications or non‑responsiveness, which are the two leading precursors to churn on Upwork and Fiverr. Early flagging enables proactive intervention before the client leaves the platform.
**A:** Integrate the `flag_for_review` flag into the platform's alert system: send an automatic notification to the account manager for any flagged review within 1 hour of submission. Schedule weekly retraining of the flagging threshold to maintain a 10% false‑positive rate.

---
## SECTION 6 — ★ BONUS TASK (10★) — Temperature A/B Test

**Business context:** Before deploying ReviewPulse's classifier in production, you need to quantify how much temperature affects output consistency on borderline reviews.

**Steps:**
1. Select 3 borderline reviews from `df` where `rating` is between **2.3 and 2.7** (use `.between()`). Take the first 3 matches.
2. Create 3 `ChatGroq` instances with `temperature=0.0`, `temperature=0.7`, and `temperature=1.0`
3. For each review, run the **same zero-shot prompt** (from Task 1) on all 3 temperature settings — **3 times each** to capture variance. Total calls: 3 reviews × 3 temps × 3 runs = 27 API calls
4. For each review + temperature combination, record the 3 predictions; compute:
   - `majority_vote`: most common prediction across 3 runs
   - `consistency_score`: how often the 3 runs agreed (e.g., 3/3 = 1.0, 2/3 = 0.67, 1/3 = 0.33)
5. Build a summary DataFrame: `review_id | temperature | majority_vote | consistency_score | true_label`
6. Print the DataFrame and state your conclusion: which temperature gives highest consistency on borderline reviews?

**Requirement:** Must use actual API calls (not hardcoded), and print the raw 3-run predictions before computing majority vote.

In [22]:
# ============================================================
# ★ BONUS — TEMPERATURE A/B TEST (corrected with raw predictions)
# Goal: Quantify how temperature affects consistency on borderline reviews.
# Method: Run zero‑shot prompt 3 times per temperature (0.0, 0.7, 1.0) on 3 borderline reviews.
# ============================================================

from collections import Counter
import re  # ensure re is imported

# --- Load GROQ_API_KEY directly (in case setup cell was skipped) ---
try:
    from google.colab import userdata
    GROQ_API_KEY = userdata.get('GROQ_API_KEY')
    print("✅ Groq API key loaded from Colab secrets.")
except (ImportError, userdata.SecretNotFoundError, userdata.NotebookAccessError) as e:
    print(f"⚠️  Could not load from secrets: {e}")
    GROQ_API_KEY = os.environ.get("GROQ_API_KEY", "YOUR_GROQ_API_KEY_HERE")
    if GROQ_API_KEY == "YOUR_GROQ_API_KEY_HERE":
        raise ValueError("❌ Please set your Groq API key in Colab secrets (name: GROQ_API_KEY) or replace the placeholder.")

# --- Select borderline reviews ---
borderline_df = df[df['rating'].between(2.3, 2.7)].head(3)
print(f"Borderline reviews selected: {len(borderline_df)}")
print(borderline_df[['review_id', 'rating', 'review_text', 'sentiment_label']])

# --- Three temperature instances (correct class: ChatGroq) ---
llm_t00 = ChatGroq(model_name="llama-3.1-8b-instant", temperature=0.0, groq_api_key=GROQ_API_KEY)
llm_t07 = ChatGroq(model_name="llama-3.1-8b-instant", temperature=0.7, groq_api_key=GROQ_API_KEY)
llm_t10 = ChatGroq(model_name="llama-3.1-8b-instant", temperature=1.0, groq_api_key=GROQ_API_KEY)

# --- Zero‑shot prompt (same as Task 1) ---
def zero_shot_prompt(review_text):
    return f"""
    Classify the following review as positive, neutral, or negative.
    Review: {review_text}
    Respond with exactly one word: positive, neutral, or negative
    """

bonus_results = []
for _, row in borderline_df.iterrows():
    print(f"\n--- Review ID: {row['review_id']} ---")
    for temp_name, llm_obj in [('0.0', llm_t00), ('0.7', llm_t07), ('1.0', llm_t10)]:
        preds = []
        for run in range(3):
            resp = llm_obj.invoke(zero_shot_prompt(row['review_text']))
            raw = resp.content.strip().lower()
            # Remove all non-alphabetic characters to avoid trailing punctuation
            clean_pred = re.sub(r'[^a-z]', '', raw)
            preds.append(clean_pred)
        print(f"  Temperature {temp_name}: raw predictions = {preds}")
        counter = Counter(preds)
        majority, count = counter.most_common(1)[0]
        consistency = count / 3
        bonus_results.append({
            'review_id': row['review_id'],
            'temperature': temp_name,
            'runs': preds,
            'majority_vote': majority,
            'consistency_score': consistency,
            'true_label': row['sentiment_label']
        })

bonus_df = pd.DataFrame(bonus_results)
print("\n=== BONUS: Temperature A/B Results ===")
print(bonus_df[['review_id', 'temperature', 'majority_vote', 'consistency_score', 'true_label']].to_string())

avg_consistency = bonus_df.groupby('temperature')['consistency_score'].mean()
print("\nAverage consistency by temperature:")
print(avg_consistency)

best_temp = avg_consistency.idxmax()
print(f"\nConclusion: temperature={best_temp} gives the highest average consistency on borderline reviews because lower temperature reduces randomness, making outputs more reproducible across runs.")

✅ Groq API key loaded from Colab secrets.
Borderline reviews selected: 3
    review_id  rating                                        review_text  \
13         14     2.7  Job was done adequately, some revisions were n...   
15         16     2.6  Work was completed as requested, nothing excep...   
20         21     2.4  Disappointed with the quality, did not meet th...   

   sentiment_label  
13         neutral  
15         neutral  
20        negative  

--- Review ID: 14 ---
  Temperature 0.0: raw predictions = ['neutral', 'neutral', 'neutral']
  Temperature 0.7: raw predictions = ['neutral', 'neutral', 'neutral']
  Temperature 1.0: raw predictions = ['neutral', 'neutral', 'neutral']

--- Review ID: 16 ---
  Temperature 0.0: raw predictions = ['neutral', 'neutral', 'neutral']
  Temperature 0.7: raw predictions = ['neutral', 'neutral', 'neutral']
  Temperature 1.0: raw predictions = ['neutral', 'neutral', 'neutral']

--- Review ID: 21 ---
  Temperature 0.0: raw predictions = ['nega

---
## SECTION 7 — Scoring Rubric

### T1 — Zero-shot vs Few-shot (15 pts)

| Component | Points | Criteria |
|-----------|--------|----------|
| Zero-shot prompt built correctly | 3 | Has instruction + format directive |
| Few-shot prompt has 3 labelled examples | 3 | One example per class |
| Both prompts end with exact one-word instruction | 2 | Exact phrase required |
| Loop runs correctly, lists populated | 3 | 10 predictions each |
| Comparison DataFrame printed with all 5 columns | 2 | review_id, text, true, zs, fs |
| Accuracy printed for both methods | 2 | Numeric values shown |

### T2 — Chain-of-Thought (20 pts)

| Component | Points | Criteria |
|-----------|--------|----------|
| CoT prompt contains "step by step" | 3 | Exact phrase |
| All 4 step labels present in prompt | 4 | 1 pt per step |
| Full LLM response printed (not truncated) | 3 | No `[:200]` or similar |
| Classification extracted programmatically | 5 | String search, not hardcoded |
| MATCH/MISMATCH verdict printed | 5 | Correct comparison to true_label |

### T3 — Role + System Prompt (20 pts)

| Component | Points | Criteria |
|-----------|--------|----------|
| Exact persona text used in system prompt | 4 | Verbatim from spec |
| User prompt requests all 4 structured keys | 4 | 1 pt per key |
| `llm.invoke()` uses tuple-list format | 4 | Not plain string |
| Both rows run and responses printed | 4 | Both neg and pos |
| Key verification loop printed for both | 4 | ✅/❌ for each key, both reviews |

### T4 — PromptTemplate + Batch JSON (20 pts)

| Component | Points | Criteria |
|-----------|--------|----------|
| `PromptTemplate` with 3 input_variables used | 4 | Correct import + instantiation |
| Prompt specifies JSON-only output | 3 | Exact instruction phrase present |
| `flag_for_review` logic in prompt (rating < 2.5) | 3 | Explicitly stated in prompt |
| `json.loads()` inside try/except | 3 | Not eval(), error handled |
| Results DataFrame with all 7 columns | 4 | Including parse_status |
| `flagged_count` printed (expected=2) | 3 | Correct value or justified if different |

### T5 — NRA Insights (15 pts)

| Component | Points | Criteria |
|-----------|--------|----------|
| Insight 1: N is single number from output | 2 | Not two numbers |
| Insight 1: R is causal, not descriptive | 1 | Mechanism explained |
| Insight 1: A is specific + committed | 2 | Named parameters |
| Insight 2: Same criteria | 5 | (2+1+2) |
| Insight 3: Same criteria | 5 | (2+1+2) |
| Internal consistency across 3 insights | -(0-5) | Contradicting actions penalised |

### ★ Bonus — Temperature A/B (10★)

| Component | Points | Criteria |
|-----------|--------|----------|
| 3 borderline reviews selected with `.between()` | 2★ | rating 2.3–2.7 |
| 3 ChatGroq instances at correct temperatures | 2★ | 0.0, 0.7, 1.0 |
| 27 API calls made (3×3×3) | 2★ | Actual calls, not hardcoded |
| Raw 3-run predictions printed per review | 2★ | Before majority vote |
| `majority_vote` + `consistency_score` computed + conclusion stated | 2★ | DataFrame + text conclusion |

---
## SECTION 9 — Interview Framing

**Q: "How do you decide between zero-shot and few-shot prompting in production?"**

> *"I evaluate three factors: task complexity, output consistency requirements, and API cost tolerance. For straightforward binary classifications I'll start with zero-shot and benchmark accuracy. If zero-shot drops below roughly 85% on a labelled validation set, I add 2–3 calibrated few-shot examples — one per class — which typically recovers 5–15 percentage points. For structured JSON extraction specifically, I always combine few-shot with a strict format directive because LLMs tend to add markdown wrappers or explanatory text without explicit instruction. In production I treat prompt versions like software versions — I track them in MLflow as prompt parameters alongside model parameters, so I can roll back if a prompt change degrades downstream accuracy."*

**Q: "When would you use Chain-of-Thought prompting?"**

> *"CoT is most valuable on borderline inputs where the LLM's direct answer is unreliable — typically edge cases in classification or multi-step reasoning tasks. It improves accuracy because forcing the model to articulate intermediate reasoning steps exposes logical errors before they become final outputs. The tradeoff is token cost and latency — a 4-step CoT response is 3–5x longer than a one-word classification. In practice I use CoT for a confidence-based routing layer: standard zero-shot for high-confidence inputs, CoT for anything that falls below a confidence threshold."*

---
**GitHub commit when done:**
```
feat: Day178 - Prompt Engineering [score/90+bonus★]
```